# GenCAD fine-tune: Qwen2.5-VL-3B, QLoRA, image -> CadQuery
Trains a LoRA adapter on a fixed subset of the `CADCODER/GenCAD-Code` train split, with the
dataset's own prompt (same prompt the baseline was scored with). Free-tier T4.

Checkpoints go to Google Drive every 50 steps. If Colab disconnects: reconnect, Run all,
and training resumes from the last checkpoint.

Afterwards, score the adapter with `gencad_baseline.ipynb`: in its cell 3 set
`MODEL_TAG = "qwen25vl3b_qlora"` and `ADAPTER = f"{OUT}/qwen_lora_adapter"`.

In [ ]:
# 1 - installs + Drive
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/gencad"
CKPT = f"{OUT}/qwen_checkpoints"
ADAPTER_DIR = f"{OUT}/qwen_lora_adapter"
os.makedirs(CKPT, exist_ok=True)


In [ ]:
# 2 - settings (change N_TRAIN here only)
N_TRAIN = 6000          # training subset size; start here, see the speed check in cell 6
MAX_CODE_TOKENS = 1500  # skip reference programs longer than this (about p95); keeps T4 memory safe
SEED = 0
BASE = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"


In [ ]:
# 3 - model + LoRA
import unsloth  # must be imported before transformers
from unsloth import FastVisionModel, is_bf16_supported
import torch

model, tokenizer = FastVisionModel.from_pretrained(BASE, load_in_4bit=True,
                                                   use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,     # images are clean renders; adapt the code-writing side
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=SEED,
)


In [ ]:
# 4 - training data: fixed subset of the train split
import json, random
from datasets import load_dataset
train = load_dataset("CADCODER/GenCAD-Code", split="train")
print("train rows:", len(train))

subset_path = f"{OUT}/train_subset_ids.json"
if os.path.exists(subset_path):
    idx = json.load(open(subset_path))
    print("reusing saved subset")
else:
    order = list(range(len(train)))
    random.Random(SEED).shuffle(order)
    idx = []
    codes = train.select_columns(["cadquery"])   # text only: skips decoding images
    for i in order:
        if len(tokenizer.tokenizer(codes[i]["cadquery"])["input_ids"]) <= MAX_CODE_TOKENS:
            idx.append(i)
        if len(idx) == N_TRAIN:
            break
    json.dump(idx, open(subset_path, "w"))
print("training subset:", len(idx), "examples")

def to_messages(row):
    # Same layout as baseline inference: image first, then the dataset prompt.
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": row["image"].convert("RGB")},
                                     {"type": "text", "text": row["prompt"]}]},
        {"role": "assistant", "content": [{"type": "text", "text": row["cadquery"]}]},
    ]}

data = [to_messages(train[i]) for i in idx]
print("converted", len(data))


In [ ]:
# 5 - trainer
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=data,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,      # 8 examples per step
        num_train_epochs=1,
        warmup_steps=20,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=10,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        output_dir=CKPT,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)
steps = len(data) // 8
print(f"{steps} optimizer steps for 1 epoch")


In [ ]:
# 6 - train (resumes from the last Drive checkpoint if there is one)
import glob, time
has_ckpt = bool(glob.glob(f"{CKPT}/checkpoint-*"))
print("resuming from checkpoint" if has_ckpt else "starting fresh")
t0 = time.time()
stats = trainer.train(resume_from_checkpoint=has_ckpt)
print(f"done in {time.time()-t0:.0f}s, mean train loss {stats.training_loss:.4f}")
# Speed check: the log prints every 10 steps. If 10 steps take over ~3 min,
# {steps} steps will not fit a free-tier day: interrupt, lower N_TRAIN in cell 2,
# delete train_subset_ids.json and qwen_checkpoints/ on Drive, Run all.


In [ ]:
# 7 - save adapter + loss log to Drive
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
log = [h for h in trainer.state.log_history if "loss" in h]
json.dump(trainer.state.log_history, open(f"{OUT}/qwen_train_log.json", "w"), indent=1)
for h in log[:: max(1, len(log) // 10)] + log[-1:]:
    print(f"step {h['step']:>5}  loss {h['loss']:.4f}")
print("adapter saved:", ADAPTER_DIR, os.listdir(ADAPTER_DIR))
